# 04 - Classification Model

Bài toán: dự đoán đơn hàng có review xấu (`review_score <= 2`) hay không. Notebook này chạy đủ ba mô hình để so sánh: Logistic Regression, Random Forest và HistGradientBoosting.


In [1]:
from src.etl.extract import load_olist_tables
from src.etl.transform import build_model_dataset, build_order_features
from src.models.train import train_model, train_model_suite

tables = load_olist_tables()
features = build_order_features(tables)
model_dataset = build_model_dataset(features)
model_dataset['bad_review'].value_counts().rename('orders').to_frame()


,orders
0,76030
1,14447


In [2]:
models, comparison = train_model_suite(model_dataset)
comparison


,model_name,n_train,n_test,positive_rate,accuracy,precision,recall,f1,roc_auc,average_precision
0,random_forest,72381,18096,0.159698,0.882239,0.683599,0.488927,0.570103,0.795595,0.589002
1,hist_gradient_boosting,72381,18096,0.159698,0.890086,0.800534,0.415225,0.546822,0.799816,0.599058
2,logistic_regression,72381,18096,0.159698,0.797580,0.396186,0.510381,0.446091,0.739653,0.469724


In [3]:
comparison_display = comparison.copy()
comparison_display['model_name'] = comparison_display['model_name'].map({
    'logistic_regression': 'Logistic Regression',
    'random_forest': 'Random Forest',
    'hist_gradient_boosting': 'HistGradientBoosting',
})
metric_cols = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'average_precision']
comparison_display[metric_cols] = comparison_display[metric_cols].round(4)
comparison_display[['model_name', *metric_cols]]


,model_name,accuracy,precision,recall,f1,roc_auc,average_precision
0,Random Forest,0.8822,0.6836,0.4889,0.5701,0.7956,0.5890
1,HistGradientBoosting,0.8901,0.8005,0.4152,0.5468,0.7998,0.5991
2,Logistic Regression,0.7976,0.3962,0.5104,0.4461,0.7397,0.4697


**Kết quả so sánh mô hình**

![Kết quả so sánh mô hình](../docs/images/model_comparison_table.png)


**Biểu đồ so sánh chỉ số mô hình**

![Biểu đồ so sánh chỉ số mô hình](../docs/images/model_comparison_metrics.png)


In [6]:
# Random Forest ???c ch?n v? ??t F1 cao nh?t tr?n t?p test.
pipeline, metrics = train_model(model_dataset, model_name='random_forest')
pd.DataFrame([{k: v for k, v in metrics.items() if k not in ['classification_report', 'confusion_matrix']}]).T.reset_index().rename(columns={'index': 'metric', 0: 'value'})


,metric,value
0,accuracy,0.8822
1,precision,0.6836
2,recall,0.4889
3,f1,0.5701
4,roc_auc,0.7956
5,average_precision,0.5890


In [7]:
pd.DataFrame(metrics['confusion_matrix'], index=['Actual 0', 'Actual 1'], columns=['Predicted 0', 'Predicted 1'])


,Predicted 0,Predicted 1
Actual 0,14552,654
Actual 1,1477,1413


**Confusion Matrix**

![Confusion Matrix](../docs/images/model_confusion_matrix.png)


**ROC Curve**

![ROC Curve](../docs/images/model_roc_curve.png)


**Precision-Recall Curve**

![Precision-Recall Curve](../docs/images/model_precision_recall_curve.png)


**Feature Importance**

![Feature Importance](../docs/images/model_feature_importance.png)
